# XAI Heatmap (ImageNet MobileNetV2)

This notebook generates an **occlusion-sensitivity heatmap** using **MobileNetV2 pretrained on ImageNet**.

- Purpose: **heatmap demo only** (not plant-disease labels).
- Works without your trained `.h5` model.
- Saves overlay PNG to Google Drive.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import cv2

from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input, decode_predictions

print('TF:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('GPU:', gpus[0].name if gpus else 'None')

## 1) Pick an image

Option A (recommended): point to your dataset folder on Drive.

Option B: set `IMG_PATH` manually to any image in Drive.

In [ ]:
DATASET_DIR = "/content/drive/MyDrive/Plant_leave_diseases_dataset_with_augmentation"  # change if needed
OUT_DIR = "/content/drive/MyDrive/models/xai_outputs_imagenet"
os.makedirs(OUT_DIR, exist_ok=True)

IMG_SIZE = (224, 224)

def find_first_image(dataset_dir):
    exts = ('.jpg', '.jpeg', '.png', '.webp')
    for root, _, files in os.walk(dataset_dir):
        for fn in files:
            if fn.lower().endswith(exts):
                return os.path.join(root, fn)
    return None

IMG_PATH = find_first_image(DATASET_DIR)
print('IMG_PATH:', IMG_PATH)
print('OUT_DIR:', OUT_DIR)

## 2) Load ImageNet model

In [ ]:
model = MobileNetV2(weights='imagenet', include_top=True, input_shape=(224, 224, 3))
model.trainable = False
print('Loaded MobileNetV2 ImageNet')

## 3) Predict (ImageNet label)

In [ ]:
img_bgr = cv2.imread(IMG_PATH)
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
img_resized = cv2.resize(img_rgb, IMG_SIZE)

x = np.expand_dims(img_resized.astype(np.float32), axis=0)
x_pp = preprocess_input(x.copy())

preds = model.predict(x_pp, verbose=0)
top = decode_predictions(preds, top=3)[0]
print('Top-3 ImageNet predictions:')
for (_, name, p) in top:
    print(f' - {name}: {p:.3f}')

target_idx = int(np.argmax(preds[0]))
target_name = top[0][1]
target_conf = float(top[0][2])

plt.figure(figsize=(5,5))
plt.imshow(img_resized.astype('uint8'))
plt.title(f"ImageNet pred: {target_name}\nConf: {target_conf:.2%}")
plt.axis('off')
plt.show()

## 4) Heatmap (Occlusion Sensitivity)

This creates a heatmap by masking patches and measuring the drop in predicted confidence for the top class.

Tune speed/quality:
- Faster: PATCH=48, STRIDE=24
- Better: PATCH=56, STRIDE=12

In [ ]:
PATCH = 48
STRIDE = 24

h, w = IMG_SIZE
base_prob = float(preds[0][target_idx])
fill = float(img_resized.mean())  # more natural than a fixed 127

rows = (h - PATCH) // STRIDE + 1
cols = (w - PATCH) // STRIDE + 1
sens = np.zeros((rows, cols), dtype=np.float32)

for ri, y in enumerate(range(0, h - PATCH + 1, STRIDE)):
    for ci, x0 in enumerate(range(0, w - PATCH + 1, STRIDE)):
        occluded = img_resized.astype(np.float32).copy()
        occluded[y:y+PATCH, x0:x0+PATCH, :] = fill
        p = model.predict(preprocess_input(np.expand_dims(occluded, 0)), verbose=0)[0]
        sens[ri, ci] = base_prob - float(p[target_idx])

sens_up = cv2.resize(sens, (w, h), interpolation=cv2.INTER_CUBIC)
sens_up = np.maximum(sens_up, 0)  # keep only confidence drops
sens_norm = (sens_up - sens_up.min()) / (sens_up.max() - sens_up.min() + 1e-8)

heat = np.uint8(255 * sens_norm)
heat_color = cv2.applyColorMap(heat, cv2.COLORMAP_JET)
overlay_bgr = cv2.addWeighted(cv2.cvtColor(img_resized.astype('uint8'), cv2.COLOR_RGB2BGR), 0.60, heat_color, 0.40, 0)
overlay_rgb = cv2.cvtColor(overlay_bgr, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(14, 4))
plt.subplot(1,3,1)
plt.imshow(img_resized.astype('uint8'))
plt.title('Original')
plt.axis('off')

plt.subplot(1,3,2)
plt.imshow(sens_norm, cmap='jet')
plt.title('XAI heatmap (occlusion)')
plt.axis('off')

plt.subplot(1,3,3)
plt.imshow(overlay_rgb)
plt.title('Overlay')
plt.axis('off')

plt.suptitle(f"ImageNet MobileNetV2 | Pred: {target_name} ({target_conf:.2%})", fontsize=11)
plt.tight_layout()

out_path = os.path.join(OUT_DIR, "xai_occlusion_imagenet_mobilenetv2.png")
plt.savefig(out_path, dpi=200, bbox_inches='tight')
plt.show()

print('Saved:', out_path)